# Run an investigation in miniature, from contract to record

*Adversarial Mathematics From First Principles*, [chapter 21](https://programmer.ie/books/math/21-chapter/) · [worked example](https://programmer.ie/examples/math/21-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

`math-engine proof <name>` reads a contract, runs its checks and controls,
decides a status and writes a record. This program does the same in
miniature with the standard library: one exact computation, one control that
must be rejected, the status rules, and a record with a manifest.

## Run it

Standard library only; nothing to install.

In [1]:
"""Running an investigation, in miniature: contract -> checks -> controls -> status -> record.

A standard-library model of what `math-engine proof <name>` does, for one computation
check. The real runner adds kernels, binding, pinned images and many more checks.
"""
import hashlib
import json
import pathlib
import tempfile


def compute(divisor):                       # the check: exact integer arithmetic
    f5 = 2 ** 32 + 1
    return "refuted" if f5 % divisor == 0 and 1 < divisor < f5 else "not refuted"


contract = {"id": "MINI-01", "check": {"divisor": 641, "expect": "refuted"},
            "controls": [{"id": "wrong-divisor", "divisor": 643, "expect": "not refuted"}],
            "evidence_kind": "computation"}

checks = [{"id": "compute", "observed": compute(contract["check"]["divisor"])}]
checks[0]["status"] = "passed" if checks[0]["observed"] == contract["check"]["expect"] else "failed"
controls = [{"id": c["id"], "observed": compute(c["divisor"])} for c in contract["controls"]]
for c, spec in zip(controls, contract["controls"]):
    c["status"] = "passed" if c["observed"] == spec["expect"] else "failed"


def status(checks, controls, all_systems, pinned, approvals, kind):
    prefix = {"kernel": "kernel_checked", "computation": "computed", "historical": "historical_verified"}[kind]
    if not all(c["status"] == "passed" for c in checks + controls):
        return "run_failed"
    if not all_systems:
        return "partial_run"
    if not pinned:
        return "checked_unpinned"
    return f"{prefix}_correspondence_{'approved' if approvals else 'pending'}"


for c in checks + controls:
    print(f"{c['id']:14} observed {c['observed']!r:15} {c['status']}")
print()
for pinned in (False, True):
    print(f"pinned toolchain {pinned!s:5} -> {status(checks, controls, True, pinned, [], contract['evidence_kind'])}")

with tempfile.TemporaryDirectory() as d:
    out = pathlib.Path(d)
    result = {"investigation": contract["id"], "checks": checks, "controls": controls,
              "status": status(checks, controls, True, True, [], contract["evidence_kind"])}
    (out / "result.json").write_text(json.dumps(result, indent=2) + "\n")
    (out / "notebook.md").write_text(f"# {contract['id']}\n\nStatus: {result['status']}\n")
    sums = "".join(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.name}\n" for p in sorted(out.iterdir()))
    (out / "SHA256SUMS").write_text(sums)
    print("\nthe record:")
    for line in sums.splitlines():
        print(f"  {line[:16]}…  {line.split('  ')[1]}")
    ok = all(hashlib.sha256((out / n).read_bytes()).hexdigest() == h
             for h, n in (l.split("  ") for l in (out / "SHA256SUMS").read_text().splitlines()))
    print(f"checking the run later (--verify): evidence intact {ok}")

compute        observed 'refuted'       passed
wrong-divisor  observed 'not refuted'   passed

pinned toolchain False -> checked_unpinned
pinned toolchain True  -> computed_correspondence_pending

the record:
  7d51e0fcf573cd5a…  notebook.md
  ca6f10f39fe7ee3d…  result.json
checking the run later (--verify): evidence intact True


## What the output shows

The check and the control both behave as the contract expects, but the
status depends on where the run happened: on unpinned tools it is only a
rehearsal (`checked_unpinned`); on the pinned toolchain, with no approval yet,
it is `computed_correspondence_pending`. The record's manifest lets the run be
checked later without re-running it.

## Not shown

- A model of the runner for one computation; the real runner also runs kernels, binds declarations to statements and records the toolchain.
- 'pinned' is a flag here; the real runner checks the image identity and tool versions.

## Challenge it

Examine the miniature run. Change the contract so that the control would pass
when it should fail, and see which status results. Then compare the program
with what `math-engine proof fermat` records in its `result.json` (Part I,
chapter 1), and list everything the real runner checks that this model does
not.